# Statistical Tables

Generates the statistical supplementary tables (S10-S36) for the BMP future performance manuscript.

## Setup

In [ ]:
"""
paths, imports, and helpers used by every section
"""
import os
import warnings
import numpy as np
import pandas as pd
from scipy.stats import (mannwhitneyu, kruskal, wilcoxon, spearmanr,
                         kendalltau, theilslopes, bootstrap)
from statsmodels.stats.contingency_tables import mcnemar
import scikit_posthocs as sp
import pymannkendall as mk

warnings.filterwarnings("ignore")

BASE   = # path to all result csvs
OUTDIR = os.path.join(BASE, "stats")

FILES = {
    "sobol_full":    os.path.join(BASE, "sobol/full_2.0/all_sobol.csv"),
    "sobol_climate": os.path.join(BASE, "sobol/full_2.0_climate/all_sobol_climate.csv"),
    "annual":        os.path.join(BASE, "6-2-all_annual_results.csv"),
    "storm_unpr":    os.path.join(BASE, "7-7-storm_analysis_dataset.csv"),
    "storm_pair":    os.path.join(BASE, "storm_pairing_dataset.csv"),
    "syn_corn":      os.path.join(BASE, "7-10-synthetic_corn_results.csv"),
    "syn_soy":       os.path.join(BASE, "7-10-synthetic_soy_results.csv"),
    "phase":         os.path.join(BASE, "7-10-phase_re_dataset.csv"),
}

REGIONS = ["a", "cp", "p", "vr"]
BMPS    = ["cc", "gb", "ma", "nm", "nt"]
SEED    = 1
N_BOOT  = 5000
rng     = np.random.default_rng(SEED)

os.makedirs(OUTDIR, exist_ok=True)

def save(table, name):
    """Write a table to OUTDIR and echo it."""
    path = os.path.join(OUTDIR, name)
    table.to_csv(path, index=False)
    print(f"\n=== {name} ===")
    print(table.to_string(index=False))

def mwu_p(a, b, min_n=3):
    """Two-sided Mann-Whitney U p-value, or NaN if either sample is too small."""
    if len(a) < min_n or len(b) < min_n:
        return np.nan
    return round(mannwhitneyu(a, b, alternative="two-sided").pvalue, 3)

def crop_failure_filter(df):
    """Drop rows where the BMP or its baseline had zero yield (YLDG + YLDF == 0)."""
    df = df.copy()
    df["crop_fail"] = (df["YLDG"] + df["YLDF"] == 0)
    keys = ["scenario", "crop", "region", "year"]
    base   = df[df["bmp"] == "base"][keys + ["crop_fail"]].rename(columns={"crop_fail": "base_fail"})
    mabase = df[df["bmp"] == "ma-base"][keys + ["crop_fail"]].rename(columns={"crop_fail": "mabase_fail"})
    df = df.merge(base, on=keys, how="left").merge(mabase, on=keys, how="left")
    baseline_fail = np.where(df["bmp"] == "ma", df["mabase_fail"], df["base_fail"])
    df["baseline_fail"] = pd.Series(baseline_fail, index=df.index).fillna(False).astype(bool)
    return df[~df["crop_fail"] & ~df["baseline_fail"]]

def re_coupling(df, tn, tp, sed, scale):
    """Per-BMP Spearman rho of TN removal against TP and sediment removal."""
    rows = []
    for bmp in BMPS:
        s = df[df["bmp"] == bmp][[tn, tp, sed]].dropna()
        if len(s) < 30:
            continue
        rho_tp, p_tp   = spearmanr(s[tn], s[tp])
        rho_sed, p_sed = spearmanr(s[tn], s[sed])
        rows.append({"scale": scale, "bmp": bmp, "n": len(s),
                     "rho_TN_TP": round(rho_tp, 3), "p_TN_TP": round(p_tp, 3),
                     "rho_TN_sed": round(rho_sed, 3), "p_TN_sed": round(p_sed, 3)})
    return pd.DataFrame(rows)

## Sobol

In [ ]:
"""
Sobol sensitivity tables.
  S14  mean S1 / ST / interaction share by parameter, climate vs operational  [full]
  S15  mean S1 by parameter x region, the Appalachia temperature flip         [climate only]
  S16  mean S1 by parameter x metric, dissolved vs erosive pathway            [climate only]
  S17  mean S2 by interaction pair, and dT x dP interaction by metric         [climate only]
  S18  dispersion of climate S1 across regions vs across crops                [climate only]
"""
climate = ["delta_t", "delta_p"]

# S14 uses the full round (includes operational parameters)
full = pd.read_csv(FILES["sobol_full"])
single = full[~full["parameter"].str.contains("x", na=False)].copy()
single["interact_share"] = single["ST"] - single["S1"]
single["param_class"] = np.where(single["parameter"].isin(climate), "climate", "operational")

s14 = (single.groupby(["parameter", "param_class"])
       .agg(mean_S1=("S1", "mean"), mean_S1_conf=("S1_conf", "mean"),
            mean_ST=("ST", "mean"), mean_interaction=("interact_share", "mean"))
       .round(3).reset_index().sort_values("mean_S1", ascending=False))
save(s14, "S14_sobol_by_parameter.csv")

# S15-S18 use the climate-only round
clim = pd.read_csv(FILES["sobol_climate"])
c_single = clim[~clim["parameter"].str.contains("x", na=False)].copy()
c_inter  = clim[clim["parameter"].str.contains("x", na=False)].copy()

s15 = (c_single.groupby(["parameter", "region"])["S1"].mean().round(3)
       .unstack("region").reset_index())
save(s15, "S15_sobol_S1_by_region.csv")

s16 = (c_single.groupby(["parameter", "metric"])["S1"].mean().round(3)
       .unstack("metric").reset_index())
save(s16, "S16_sobol_S1_by_metric.csv")

s17_pairs = (c_inter.groupby("parameter")["S2"].mean().round(3)
             .sort_values(ascending=False).reset_index(name="mean_S2"))
save(s17_pairs, "s17a_sobol_S2_by_pair.csv")

txp = c_inter[c_inter["parameter"] == "delta_t x delta_p"]
s17_txp = txp.groupby("metric")["S2"].mean().round(3).reset_index(name="mean_S2_TxP")
save(s17_txp, "s17b_sobol_TxP_by_metric.csv")

# S18 needs a crop column in the Sobol output; indices are run per region-crop-BMP
climate_only = c_single[c_single["parameter"].isin(climate)]
across_region = climate_only.groupby(["parameter", "region"])["S1"].mean().groupby("parameter").std()
across_crop   = climate_only.groupby(["parameter", "crop"])["S1"].mean().groupby("parameter").std()
s18 = pd.DataFrame({"std_across_regions": across_region.round(3),
                    "std_across_crops":   across_crop.round(3)}).reset_index()
s18["region_to_crop_ratio"] = (s18["std_across_regions"] / s18["std_across_crops"]).round(1)
save(s18, "S18_sobol_dispersion_region_vs_crop.csv")


## Annual - baseline loads

In [ ]:
"""
Annual baseline (no-BMP) load tables, pooled across crops.
  S19  Mann-Kendall p and Theil-Sen slope for baseline TN/TP/sediment, both RCPs
  S20  baseline period comparisons: medians, percent change, Mann-Whitney U
"""
LOADS = ["TN", "TP", "MUSL"]
LABEL = {"reference period": "hist", "RCP 45": "RCP45", "RCP 85": "RCP85"}

df = pd.read_csv(FILES["annual"])
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")], errors="ignore")
base = df[df["bmp"] == "base"].copy()

def pool(sub):
    return sub.groupby(["region", "year"])[LOADS].mean().reset_index()

# S19: trend over the future window per RCP
rows = []
for rcp in ["RCP 45", "RCP 85"]:
    g = pool(base[base["scenario"] == rcp])
    for reg in REGIONS:
        sub = g[g["region"] == reg].sort_values("year")
        row = {"scenario": LABEL[rcp], "region": reg}
        for m in LOADS:
            row[f"{m}_mk_p"]  = round(mk.original_test(sub[m].values).p, 3)
            row[f"{m}_slope"] = round(theilslopes(sub[m].values, sub["year"].values)[0], 3)
        rows.append(row)
save(pd.DataFrame(rows), "S19_baseline_load_trends.csv")

# S20: period comparisons
def period_slice(scenario, lo=None, hi=None):
    s = base[base["scenario"] == scenario]
    if lo is not None:
        s = s[s["year"].between(lo, hi)]
    return pool(s)

periods = {"hist":   period_slice("reference period"),
           "early":  period_slice("RCP 45", 2021, 2050),
           "late45": period_slice("RCP 45", 2051, 2090),
           "late85": period_slice("RCP 85", 2051, 2090)}

rows = []
for reg in REGIONS:
    for m in LOADS:
        vals = {k: v[v["region"] == reg][m].values for k, v in periods.items()}
        med  = {k: np.median(x) for k, x in vals.items()}
        rows.append({
            "region": reg, "metric": m,
            "hist_med": round(med["hist"], 2),   "early_med": round(med["early"], 2),
            "late45_med": round(med["late45"], 2), "late85_med": round(med["late85"], 2),
            "pct_chg_late45": round(100 * (med["late45"] - med["hist"]) / med["hist"], 1),
            "pct_chg_late85": round(100 * (med["late85"] - med["hist"]) / med["hist"], 1),
            "p_hist_early":   mwu_p(vals["hist"], vals["early"]),
            "p_early_late45": mwu_p(vals["early"], vals["late45"]),
            "p_hist_late85":  mwu_p(vals["hist"], vals["late85"]),
        })
save(pd.DataFrame(rows), "S20_baseline_period_comparisons.csv")


## Annual - removal efficiency

In [ ]:
"""
Annual BMP removal-efficiency tables (crop-failure filtered, median across crops).
  S21  Mann-Kendall p and Theil-Sen slope for RE, both RCPs
  S22  RE period comparisons: medians and Mann-Whitney U
"""
RE = ["RE_TN", "RE_TP", "RE_MUSL"]

df = pd.read_csv(FILES["annual"])
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")], errors="ignore")

hist = crop_failure_filter(df[df["scenario"] == "reference period"]).query("bmp in @BMPS")
fut  = crop_failure_filter(df[df["scenario"] != "reference period"]).query("bmp in @BMPS")

agg_hist = hist.groupby(["region", "bmp", "year"])[RE].median().reset_index()
agg_fut  = fut.groupby(["scenario", "region", "bmp", "year"])[RE].median().reset_index()

# S21: trend per RCP
rows = []
for rcp in ["RCP 45", "RCP 85"]:
    for reg in REGIONS:
        for bmp in BMPS:
            sub = agg_fut.query("scenario == @rcp and region == @reg and bmp == @bmp").sort_values("year")
            row = {"scenario": rcp, "region": reg, "bmp": bmp}
            for m in RE:
                s = sub[["year", m]].dropna()
                if len(s) < 5:
                    row[f"{m}_p"], row[f"{m}_slope"] = np.nan, np.nan
                else:
                    row[f"{m}_p"]     = round(mk.original_test(s[m].values).p, 3)
                    row[f"{m}_slope"] = round(theilslopes(s[m].values, s["year"].values)[0], 3)
            rows.append(row)
save(pd.DataFrame(rows), "S21_RE_trends.csv")

# S22: period comparisons
rows = []
for reg in REGIONS:
    for bmp in BMPS:
        for m in RE:
            h   = agg_hist.query("region == @reg and bmp == @bmp")[m].dropna().values
            e   = agg_fut.query("scenario == 'RCP 45' and region == @reg and bmp == @bmp and year <= 2050")[m].dropna().values
            l45 = agg_fut.query("scenario == 'RCP 45' and region == @reg and bmp == @bmp and year > 2050")[m].dropna().values
            l85 = agg_fut.query("scenario == 'RCP 85' and region == @reg and bmp == @bmp and year > 2050")[m].dropna().values
            rows.append({
                "region": reg, "bmp": bmp, "metric": m,
                "hist_med":   round(np.median(h), 1) if len(h) else np.nan,
                "early_med":  round(np.median(e), 1) if len(e) else np.nan,
                "late45_med": round(np.median(l45), 1) if len(l45) else np.nan,
                "late85_med": round(np.median(l85), 1) if len(l85) else np.nan,
                "p_h_e":   mwu_p(h, e),   "p_h_l45": mwu_p(h, l45),
                "p_h_l85": mwu_p(h, l85), "p_e_l45": mwu_p(e, l45),
            })
save(pd.DataFrame(rows), "S22_RE_period_comparisons.csv")


## Crop failure & filtering

In [ ]:
"""
Crop-failure and filtering diagnostics.
  S11  zero-yield failure rate by BMP and period
  S12  effect of the crop-failure filter on reported RE medians
  S13  Piedmont cover-crop failure rate by decade and RCP
"""
RE = ["RE_TN", "RE_TP", "RE_MUSL"]

df = pd.read_csv(FILES["annual"])
df = df.drop(columns=[c for c in df.columns if c.startswith("Unnamed")], errors="ignore")
df["crop_fail"] = (df["YLDG"] + df["YLDF"] == 0)
df["period"] = np.where(df["scenario"] == "reference period", "historical",
               np.where(df["year"] <= 2050, "early",
               np.where(df["scenario"] == "RCP 45", "late_RCP45", "late_RCP85")))

# S11: failure rate by BMP x period (before filtering, so failures are still counted)
s11 = (df.groupby(["bmp", "period"])["crop_fail"].mean().round(3)
       .unstack("period").reset_index())
save(s11, "S11_crop_failure_rates.csv")

# S12: filter effect on medians, unfiltered vs filtered
kept = crop_failure_filter(df).query("bmp in @BMPS")
rows = []
for bmp in BMPS:
    for m in RE:
        full_vals = df.query("bmp == @bmp")[m].dropna().values
        keep_vals = kept.query("bmp == @bmp")[m].dropna().values
        rows.append({"bmp": bmp, "metric": m,
                     "median_unfiltered": round(np.median(full_vals), 2),
                     "median_filtered":   round(np.median(keep_vals), 2),
                     "delta":             round(np.median(keep_vals) - np.median(full_vals), 2),
                     "mwu_p":             mwu_p(full_vals, keep_vals)})
save(pd.DataFrame(rows), "S12_filter_effect_on_medians.csv")

# S13: Piedmont cover-crop failure rate by decade and RCP
pc = df.query("region == 'p' and bmp == 'cc' and scenario != 'reference period'").copy()
pc["decade"] = (pc["year"] // 10) * 10
s13 = (pc.groupby(["scenario", "decade"])["crop_fail"].mean().round(3)
       .unstack("scenario").reset_index())
save(s13, "S13_piedmont_cc_failure_by_decade.csv")


## Storm analysis

In [ ]:
"""
Event-scale storm tables.
  S23  pollutant coupling: RE_TN vs RE_sed and RE_TP [unpaired]
  S24  storm characteristics by region x period            [paired dataset, descriptive]
  S25  baseline load vs runoff / depth / return period      [unpaired]
  S26  load-weighted RE with bootstrap CIs by scenario       [unpaired]
  S27  paired storm change with Wilcoxon and McNemar tests   [paired]
"""
POLL     = ["TN", "TP", "sediment"]
SCEN     = ["historical", "RCP45", "RCP85"]
PERIODS  = ["hist", "early", "late_RCP45", "late_RCP85"]
BASE_BMP = "gb"   # storm characteristics and baseline load are BMP-independent; gb exists for every crop

up = pd.read_csv(FILES["storm_unpr"])
pr = pd.read_csv(FILES["storm_pair"])

def assign_period(d):
    def per(r):
        if r["scenario"] == "historical":
            return "hist"
        if r["sim_year"] <= 2050:
            return "early" if r["scenario"] == "RCP45" else "drop"
        return "late_" + r["scenario"]
    d = d.copy()
    d["period"] = d.apply(per, axis=1)
    return d[d["period"] != "drop"]

# S23: pollutant coupling
up_bmps = up[up["bmp"].isin(BMPS)]
s23 = re_coupling(up_bmps, "re_TN", "re_TP", "re_sediment", "storm")
save(s23, "S23_metric_coupling_storm.csv")

# S24: storm characteristics
g = assign_period(up)
g = g[g["bmp"] == BASE_BMP]
rows = []
for reg in REGIONS:
    for p in PERIODS:
        s = g[(g["region"] == reg) & (g["period"] == p)]
        rows.append({"region": reg, "period": p, "n_events": len(s),
                     "peak_24h_med":      round(s["peak_24h"].median(), 1) if len(s) else np.nan,
                     "depth_med":         round(s["event_total_depth"].median(), 1) if len(s) else np.nan,
                     "runoff_med":        round(s["event_runoff_Q"].median(), 2) if len(s) else np.nan,
                     "pct_with_runoff":   round(100 * (s["event_runoff_Q"] > 0).mean(), 1) if len(s) else np.nan,
                     "pct_productive_TN": round(100 * (s["baseline_TN"] > 0).mean(), 1) if len(s) else np.nan})
save(pd.DataFrame(rows), "S24_storm_characteristics.csv")

# S25: baseline load correlations
base = up[up["bmp"] == BASE_BMP]
rows = []
for reg in REGIONS:
    s = base[base["region"] == reg]
    for load in [f"baseline_{p}" for p in POLL]:
        for x in ["event_runoff_Q", "event_total_depth", "return_period"]:
            d = s[[load, x]].dropna()
            rho = round(spearmanr(d[x], d[load])[0], 2) if len(d) >= 30 else np.nan
            rows.append({"region": reg, "load": load, "predictor": x, "spearman_rho": rho, "n": len(d)})
save(pd.DataFrame(rows), "S25_baseline_load_correlations.csv")

# S26: load-weighted RE with bootstrap CIs
def lwre(b, m):
    return 100 * (b.sum() - m.sum()) / b.sum()

def lwre_boot(b, m):
    res = bootstrap((b, m), lwre, paired=True, n_resamples=N_BOOT,
                    random_state=rng, method="percentile", vectorized=False)
    return res.confidence_interval.low, res.confidence_interval.high, res.bootstrap_distribution

rows = []
for poll in POLL:
    bcol, mcol = f"baseline_{poll}", f"bmp_{poll}"
    d0 = up[up[bcol] > 0].dropna(subset=[bcol, mcol])
    for bmp in BMPS:
        row = {"pollutant": poll, "bmp": bmp}
        pt, dist = {}, {}
        for sc in SCEN:
            d = d0[(d0["bmp"] == bmp) & (d0["scenario"] == sc)]
            b, m = d[bcol].to_numpy(), d[mcol].to_numpy()
            if len(b) < 2:
                pt[sc], dist[sc], row[sc] = np.nan, None, "na"
                continue
            pt[sc] = lwre(b, m)
            lo, hi, dist[sc] = lwre_boot(b, m)
            row[sc] = f"{pt[sc]:.1f} [{lo:.1f}, {hi:.1f}]"
        if dist["historical"] is not None and dist["RCP85"] is not None:
            diff = dist["historical"] - dist["RCP85"]
            dlo, dhi = np.percentile(diff, [2.5, 97.5])
            flag = "sig" if (dlo > 0 or dhi < 0) else "ns"
            row["change_hist_to_RCP85"] = f"{pt['historical'] - pt['RCP85']:.1f} [{dlo:.1f}, {dhi:.1f}] {flag}"
        else:
            row["change_hist_to_RCP85"] = "na"
        rows.append(row)
save(pd.DataFrame(rows), "S26_loadweighted_RE.csv")

# S27: paired storm change and tests
g = assign_period(pr)
g = g[g["bmp"] == BASE_BMP]

def per_storm(metric):
    return g.groupby(["storm_id", "period"])[metric].median().unstack("period")

rows = []
for p in PERIODS:
    s = g[g["period"] == p]
    rows.append({"period": p, "n_events": len(s),
                 "peak_med":  round(s["peak_24h"].median(), 1),
                 "depth_med": round(s["event_total_depth"].median(), 1),
                 "runoff_med": round(s["event_runoff_Q"].median(), 2),
                 "pct_productive_TN": round(100 * (s["baseline_TN"] > 0).mean(), 1)})
d1 = pd.DataFrame(rows)
h = d1[d1["period"] == "hist"].iloc[0]
d1["peak_pct_vs_hist"]  = round((d1["peak_med"] / h["peak_med"] - 1) * 100, 1)
d1["depth_pct_vs_hist"] = round((d1["depth_med"] / h["depth_med"] - 1) * 100, 1)
save(d1, "S27a_storm_change_by_period.csv")

rows = []
for late in ["late_RCP45", "late_RCP85"]:
    for metric in ["peak_24h", "event_total_depth", "event_runoff_Q"]:
        w = per_storm(metric)[["hist", late]].dropna()
        p = wilcoxon(w["hist"], w[late]).pvalue
        rows.append({"comparison": f"hist_vs_{late}", "metric": metric, "test": "Wilcoxon",
                     "median_change": round((w[late] - w["hist"]).median(), 3),
                     "p_value": f"{p:.2e}", "n_pairs": len(w)})
    prod = g.groupby(["storm_id", "period"])["baseline_TN"].median().unstack("period")[["hist", late]].dropna()
    hy, ly = prod["hist"] > 0, prod[late] > 0
    table = [[int((hy & ly).sum()),  int((hy & ~ly).sum())],
             [int((~hy & ly).sum()), int((~hy & ~ly).sum())]]
    p = mcnemar(table, exact=False, correction=True).pvalue
    rows.append({"comparison": f"hist_vs_{late}", "metric": "productive_TN_fraction", "test": "McNemar",
                 "median_change": f"b={table[0][1]}, c={table[1][0]}", "p_value": f"{p:.2e}",
                 "n_pairs": len(prod)})
save(pd.DataFrame(rows), "S27b_storm_paired_tests.csv")


## Synthetic storm

In [ ]:
"""
Synthetic-storm tables (corn and soy).
  S28  return-period robustness (Spearman, Kendall tau)
  S29  warming (dT) sensitivity of RE
  S30  timing (window) pairwise tests, and net-export fractions
  S31  pollutant coupling: RE_TN vs RE_sed and RE_TP
  S32  composition: sediment fraction and soluble removal for N and P
  S33  nitrate coupling: RE_TN vs root-zone nitrate, with cover-crop terciles
  S34  regional postharvest RE_TN, and warming effect on soil pools
"""
POLL_RE = {"RE_TN": "TN", "RE_TP": "TP", "RE_sed": "sed"}
WINDOWS = ["preplant", "midseason", "postharvest"]
SYN_REG = ["appalachia", "coastal_plain", "piedmont", "valley_ridge"]
POOLS   = ["ZNO3", "ZNH3", "ZPML"]
DT_LO, DT_HI = 0.0, 2.7
MIN_N   = 3

def load_syn(path):
    d = pd.read_csv(path)
    return d[d["undefined"] == False].copy()

DATA = {"corn": load_syn(FILES["syn_corn"]), "soy": load_syn(FILES["syn_soy"])}

def real(d):
    return d[d["bmp"].isin(BMPS)]

# S28: return-period robustness
rows = []
for crop, df in DATA.items():
    r = real(df)
    for bmp in BMPS:
        for col, lab in POLL_RE.items():
            s = r[r["bmp"] == bmp][["return_period", col]].dropna()
            if len(s) < 6:
                continue
            rho, ps = spearmanr(s["return_period"], s[col])
            tau, pt = kendalltau(s["return_period"], s[col])
            rows.append({"crop": crop, "bmp": bmp, "pollutant": lab, "n": len(s),
                         "spearman_rho": round(rho, 3), "spearman_p": round(ps, 3),
                         "kendall_tau": round(tau, 3), "kendall_p": round(pt, 3),
                         "sig": (ps < 0.05) or (pt < 0.05)})
save(pd.DataFrame(rows), "S28_synthetic_return_period.csv")

# S29: warming sensitivity
rows = []
for crop, df in DATA.items():
    r = real(df)
    for bmp in BMPS:
        for col, lab in POLL_RE.items():
            s = r[r["bmp"] == bmp][["dT", col]].dropna()
            a = s[s["dT"] == DT_LO][col]
            b = s[s["dT"] == DT_HI][col]
            if len(a) < MIN_N or len(b) < MIN_N:
                continue
            pu = mannwhitneyu(a, b).pvalue
            rho, ps = spearmanr(s["dT"], s[col])
            rows.append({"crop": crop, "bmp": bmp, "pollutant": lab,
                         "med_hist": round(a.median(), 3), "med_warm": round(b.median(), 3),
                         "delta": round(b.median() - a.median(), 3),
                         "mwu_p": round(pu, 3), "spearman_rho": round(rho, 3),
                         "spearman_p": round(ps, 3), "sig": (pu < 0.05) or (ps < 0.05)})
save(pd.DataFrame(rows), "S29_synthetic_warming.csv")

# S30: timing pairwise tests and net-export fractions
rows = []
for crop, df in DATA.items():
    r = real(df)
    for bmp in BMPS:
        meds = {w: r[(r["bmp"] == bmp) & (r["window"] == w)]["RE_TN"].median() for w in WINDOWS}
        for i in range(len(WINDOWS)):
            for j in range(i + 1, len(WINDOWS)):
                a = r[(r["bmp"] == bmp) & (r["window"] == WINDOWS[i])]["RE_TN"].dropna()
                b = r[(r["bmp"] == bmp) & (r["window"] == WINDOWS[j])]["RE_TN"].dropna()
                if len(a) < MIN_N or len(b) < MIN_N:
                    continue
                pu = mannwhitneyu(a, b).pvalue
                rows.append({"crop": crop, "bmp": bmp, "pair": f"{WINDOWS[i]}_vs_{WINDOWS[j]}",
                             "med_a": round(meds[WINDOWS[i]], 2), "med_b": round(meds[WINDOWS[j]], 2),
                             "mwu_p": round(pu, 3), "sig": pu < 0.05})
save(pd.DataFrame(rows), "S30a_synthetic_timing.csv")

rows = []
for crop, df in DATA.items():
    r = real(df)
    for bmp in BMPS:
        row = {"crop": crop, "bmp": bmp}
        for w in WINDOWS:
            v = r[(r["bmp"] == bmp) & (r["window"] == w)]["RE_TN"].dropna()
            row[f"frac_neg_{w}"] = round((v < 0).mean(), 3) if len(v) else np.nan
        rows.append(row)
save(pd.DataFrame(rows), "S30b_synthetic_net_export.csv")

# S31: pollutant coupling
rows = []
for crop, df in DATA.items():
    r = real(df)
    for bmp in BMPS:
        s = r[r["bmp"] == bmp][["RE_TN", "RE_sed", "RE_TP"]].dropna()
        if len(s) < 5:
            continue
        rho_sed, p_sed = spearmanr(s["RE_TN"], s["RE_sed"])
        rho_tp, p_tp   = spearmanr(s["RE_TN"], s["RE_TP"])
        rows.append({"crop": crop, "bmp": bmp, "n": len(s),
                     "rho_TN_sed": round(rho_sed, 3), "p_TN_sed": round(p_sed, 3),
                     "rho_TN_TP": round(rho_tp, 3), "p_TN_TP": round(p_tp, 3)})
save(pd.DataFrame(rows), "S31_synthetic_coupling.csv")

# S32: composition for N and P (soluble = Q, sediment-bound = Y)
def composition(nutrient):
    q, y = f"Q{nutrient}", f"Y{nutrient}"
    key = ["region", "window", "return_period", "dT"]
    out = []
    for crop, df in DATA.items():
        r = real(df).copy()
        base   = df[df["bmp"] == "base"][key + [q]].rename(columns={q: "qb"})
        mabase = df[df["bmp"] == "ma-base"][key + [q]].rename(columns={q: "qmab"})
        for bmp in BMPS:
            s = r[r["bmp"] == bmp].merge(base, on=key, how="left").merge(mabase, on=key, how="left")
            s["sedfrac"] = s[y] / (s[y] + s[q]).replace(0, np.nan)
            s["qbase"] = np.where(bmp == "ma", s["qmab"], s["qb"])
            s = s[s["qbase"] > 0].copy()
            s["RE_soluble"] = 1 - s[q] / s["qbase"]
            sol = s["RE_soluble"].replace([np.inf, -np.inf], np.nan).dropna()
            out.append({"crop": crop, "bmp": bmp, "nutrient": nutrient,
                        "med_sed_frac": round(s["sedfrac"].median(), 3),
                        "med_RE_soluble": round(sol.median(), 3) if len(sol) else np.nan})
    return out

s32 = pd.DataFrame(composition("N") + composition("P"))
save(s32, "S32_synthetic_composition.csv")

# S33: nitrate coupling and cover-crop terciles
rows = []
for crop, df in DATA.items():
    r = real(df)
    for bmp in BMPS:
        s = r[r["bmp"] == bmp][["RE_TN", "ZNO3"]].dropna()
        if len(s) < 6:
            continue
        rho, p = spearmanr(s["ZNO3"], s["RE_TN"])
        rows.append({"crop": crop, "bmp": bmp, "n": len(s),
                     "rho_RE_vs_nitrate": round(rho, 3), "p": round(p, 3), "sig": p < 0.05})
save(pd.DataFrame(rows), "S33a_nitrate_coupling.csv")

rows = []
for crop, df in DATA.items():
    r = real(df)
    for bmp in ["cc", "gb", "ma"]:
        s = r[r["bmp"] == bmp][["RE_TN", "ZNO3"]].dropna()
        if len(s) < 9:
            continue
        s = s.copy()
        s["tercile"] = pd.qcut(s["ZNO3"].rank(method="first"), 3, labels=["low", "mid", "high"])
        for t, grp in s.groupby("tercile", observed=True):
            rows.append({"crop": crop, "bmp": bmp, "nitrate_tercile": t, "n": len(grp),
                         "nitrate_median": round(grp["ZNO3"].median(), 4),
                         "frac_net_source": round((grp["RE_TN"] < 0).mean(), 3),
                         "med_RE_TN": round(grp["RE_TN"].median(), 3)})
save(pd.DataFrame(rows), "S33b_nitrate_terciles.csv")

# S34: regional postharvest RE_TN, and warming effect on baseline soil pools
rows = []
for crop, df in DATA.items():
    r = real(df)
    for bmp in BMPS:
        row = {"crop": crop, "bmp": bmp}
        for reg in SYN_REG:
            v = r[(r["bmp"] == bmp) & (r["window"] == "postharvest") & (r["region"] == reg)]["RE_TN"].dropna()
            row[reg] = round(v.median(), 3) if len(v) else np.nan
        rows.append(row)
save(pd.DataFrame(rows), "S34a_regional_postharvest.csv")

rows = []
for crop, df in DATA.items():
    b = df[df["bmp"] == "base"]
    for pool in POOLS:
        a = b[b["dT"] == DT_LO][pool].dropna()
        c = b[b["dT"] == DT_HI][pool].dropna()
        if len(a) < MIN_N or len(c) < MIN_N:
            continue
        pu = mannwhitneyu(a, c).pvalue
        rows.append({"crop": crop, "pool": pool,
                     "med_hist": round(a.median(), 4), "med_warm": round(c.median(), 4),
                     "delta": round(c.median() - a.median(), 4),
                     "mwu_p": round(pu, 3), "sig": pu < 0.05})
save(pd.DataFrame(rows), "S35b_warming_pools.csv")


## Operational phase

In [ ]:
"""
Operational-phase tables (corn and soy, RCP4.5 and RCP8.5 pooled).
  S35 pollutant coupling: RE_TN vs RE_sed and RE_TP
  S36 phase-level effect: Kruskal-Wallis, medians, Dunn post-hoc
  S37 temporal trend by phase: Theil-Sen slope, Mann-Kendall, region agreement
  S38 RCP pooling check: Mann-Whitney RCP4.5 vs RCP8.5
"""
METRIC = "re_TN"
PERIOD = (2021, 2090)
PHASES = ["preplant", "midseason", "postharvest"]
KEYS   = ["scenario", "region", "crop", "bmp", "phase_year", "phase"]

df = pd.read_csv(FILES["phase"])
dup = df.duplicated(subset=KEYS, keep=False)
df = df[~(dup & (df["bmp_crop_failure"] == 1))].drop_duplicates(subset=KEYS)
df = df[(df["baseline_crop_failure"] == 0) & (df["bmp_crop_failure"] == 0)]
d = df[df["scenario"].isin(["RCP45", "RCP85"]) & df["phase_year"].between(*PERIOD)]

# S35: pollutant coupling
df_bmps = df[df["bmp"].isin(BMPS)]
s35 = re_coupling(df_bmps, "re_TN", "re_TP", "re_sediment", "phase")
save(s35, "S35_metric_coupling_phase.csv")

# S36: phase-level effect
rows = []
for b in BMPS:
    sub = d[d["bmp"] == b]
    groups = [sub[sub["phase"] == p][METRIC].dropna() for p in PHASES]
    H, p = kruskal(*groups)
    dunn = sp.posthoc_dunn(sub, val_col=METRIC, group_col="phase")
    rows.append({"bmp": b, "KW_H": round(H, 1), "KW_p": f"{p:.2e}",
                 **{f"med_{ph[:3]}": round(sub[sub["phase"] == ph][METRIC].median(), 1) for ph in PHASES},
                 "dunn_pre_mid":  round(dunn.loc["preplant", "midseason"], 4),
                 "dunn_pre_post": round(dunn.loc["preplant", "postharvest"], 4),
                 "dunn_mid_post": round(dunn.loc["midseason", "postharvest"], 4)})
save(pd.DataFrame(rows), "S36_phase_effect.csv")

# S37: temporal trend (pooled-RCP median series) with region agreement
rows = []
for b in BMPS:
    for ph in PHASES:
        sub = d[(d["bmp"] == b) & (d["phase"] == ph)]
        ser = sub.groupby("phase_year")[METRIC].median().sort_index()
        res = mk.original_test(ser.values)
        slope = theilslopes(ser.values, ser.index.values)[0] * 10
        ndecl = 0
        for _, gg in sub.groupby("region"):
            s2 = gg.groupby("phase_year")[METRIC].median().sort_index()
            if s2.nunique() < 2 or len(s2) < 10:
                continue
            if mk.original_test(s2.values).p < 0.05 and theilslopes(s2.values, s2.index.values)[0] < 0:
                ndecl += 1
        rows.append({"bmp": b, "phase": ph, "slope_pp_dec": round(slope, 2),
                     "MK_p": round(res.p, 4), "trend": res.trend, "regions_declining": f"{ndecl}/4"})
save(pd.DataFrame(rows), "S37_phase_trends.csv")

# S38: RCP pooling check
rows = []
for b in BMPS:
    for ph in PHASES:
        a = d[(d["bmp"] == b) & (d["phase"] == ph) & (d["scenario"] == "RCP45")].groupby("phase_year")[METRIC].median()
        c = d[(d["bmp"] == b) & (d["phase"] == ph) & (d["scenario"] == "RCP85")].groupby("phase_year")[METRIC].median()
        if len(a) < 3 or len(c) < 3:
            continue
        p = mannwhitneyu(a, c).pvalue
        rows.append({"bmp": b, "phase": ph, "MW_p": round(p, 3),
                     "slope45": round(theilslopes(a.values, a.index.values)[0] * 10, 2),
                     "slope85": round(theilslopes(c.values, c.index.values)[0] * 10, 2),
                     "differ": p < 0.05})
save(pd.DataFrame(rows), "S38_rcp_pooling.csv")


## Compile workbook

In [ ]:
"""
Compile every CSV in OUTDIR into a single Excel workbook, one sheet per table.
The individual CSVs are left untouched; this only writes a combined copy.
"""
import glob

WORKBOOK = os.path.join(OUTDIR, "all_stats_tables.xlsx")

def sheet_name(path):
    name = os.path.splitext(os.path.basename(path))[0]
    for ch in "[]:*?/\\":
        name = name.replace(ch, "_")
    return name[:31]

csvs = sorted(glob.glob(os.path.join(OUTDIR, "*.csv")))
with pd.ExcelWriter(WORKBOOK, engine="openpyxl") as writer:
    for path in csvs:
        pd.read_csv(path).to_excel(writer, sheet_name=sheet_name(path), index=False)
print(f"wrote {WORKBOOK} with {len(csvs)} sheets")
